<a href="https://colab.research.google.com/github/neel20409/DataScience/blob/main/project02_iris.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

print("✅ All libraries imported!")

In [ ]:
# Load built-in iris dataset — no download needed!
iris = load_iris()

# Convert to DataFrame for easy exploration
df = pd.DataFrame(iris.data, columns=iris.feature_names)

# Add the target column (species)
df['species'] = iris.target
df['species_name'] = df['species'].map({0:'setosa', 1:'versicolor', 2:'virginica'})

print(df.shape)       # (150, 6) — 150 flowers, 6 columns
df.head(10)

In [ ]:
# Basic info
print(df.info())
print("\n")
print(df.describe())

# How many flowers of each species?
print(df['species_name'].value_counts())
# Output: 50 of each — perfectly balanced dataset!

# Any missing values?
print(df.isnull().sum())
# Output: all zeros — clean dataset!

In [ ]:
# 1. Count of each species
plt.figure(figsize=(6,4))
sns.countplot(x='species_name', data=df, palette='Set2')
plt.title("Flower Count by Species")
plt.show()

# 2. Pairplot — see all features vs all features at once
sns.pairplot(df, hue='species_name', palette='Set2')
plt.suptitle("Pairplot of Iris Features", y=1.02)
plt.show()
# This one chart tells you EVERYTHING about the dataset

# 3. Heatmap — correlation between features
plt.figure(figsize=(8,5))
sns.heatmap(df.corr(numeric_only=True), annot=True,
            cmap='coolwarm', fmt='.2f')
plt.title("Feature Correlation Heatmap")
plt.show()

# 4. Boxplot — distribution of each feature by species
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
features = ['sepal length (cm)', 'sepal width (cm)',
            'petal length (cm)', 'petal width (cm)']

for i, feature in enumerate(features):
    ax = axes[i//2, i%2]
    sns.boxplot(x='species_name', y=feature, data=df,
                palette='Set2', ax=ax)
    ax.set_title(f'{feature} by Species')

plt.tight_layout()
plt.show()

In [ ]:
# X = features (input) — what the model learns FROM
X = df[['sepal length (cm)', 'sepal width (cm)',
        'petal length (cm)', 'petal width (cm)']]

# y = target (output) — what the model predicts
y = df['species']

print("X shape:", X.shape)  # (150, 4) — 150 rows, 4 features
print("y shape:", y.shape)  # (150,) — 150 labels

# Split into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,      # 20% for testing, 80% for training
    random_state=42,    # makes results reproducible
    stratify=y          # keeps equal species ratio in both sets
)

print(f"\nTraining samples: {X_train.shape[0]}")  # 120
print(f"Testing samples:  {X_test.shape[0]}")     # 30

In [ ]:
# Define all 4 models
models = {
    'Logistic Regression': LogisticRegression(max_iter=200),
    'Decision Tree':       DecisionTreeClassifier(random_state=42),
    'Random Forest':       RandomForestClassifier(random_state=42),
    'KNN':                 KNeighborsClassifier(n_neighbors=5)
}

results = {}

# Train and evaluate each model
for name, model in models.items():
    # Train — model learns from training data
    model.fit(X_train, y_train)

    # Predict — model predicts on test data
    y_pred = model.predict(X_test)

    # Score — how accurate is it?
    acc = accuracy_score(y_test, y_pred)
    results[name] = acc
    print(f"{name}: {acc*100:.2f}% accuracy")

In [ ]:
# Bar chart comparing all models
plt.figure(figsize=(8,5))
models_names = list(results.keys())
accuracies = [v*100 for v in results.values()]

bars = plt.bar(models_names, accuracies, color=['#2ecc71','#3498db','#e74c3c','#f39c12'])
plt.ylim(90, 102)
plt.ylabel("Accuracy (%)")
plt.title("Model Accuracy Comparison")

# Add accuracy labels on top of bars
for bar, acc in zip(bars, accuracies):
    plt.text(bar.get_x() + bar.get_width()/2,
             bar.get_height() + 0.2,
             f'{acc:.1f}%', ha='center', fontsize=11)

plt.tight_layout()
plt.show()

In [ ]:
# Use Random Forest as our best model
best_model = RandomForestClassifier(random_state=42)
best_model.fit(X_train, y_train)
y_pred = best_model.predict(X_test)

# 1. Classification Report — detailed breakdown
print("Classification Report:")
print(classification_report(y_test, y_pred,
      target_names=['setosa','versicolor','virginica']))

# 2. Confusion Matrix — see exactly where model went wrong
plt.figure(figsize=(6,5))
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['setosa','versicolor','virginica'],
            yticklabels=['setosa','versicolor','virginica'])
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix — Random Forest")
plt.show()

# 3. Feature Importance — which feature matters most?
feature_importance = pd.DataFrame({
    'feature': X.columns,
    'importance': best_model.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(7,4))
sns.barplot(x='importance', y='feature',
            data=feature_importance, palette='viridis')
plt.title("Feature Importance — Which feature matters most?")
plt.show()

print(feature_importance)

In [ ]:
# Predict a brand new flower the model has never seen!
new_flower = pd.DataFrame([[5.1, 3.5, 1.4, 0.2]],
             columns=['sepal length (cm)', 'sepal width (cm)',
                      'petal length (cm)', 'petal width (cm)'])

prediction = best_model.predict(new_flower)
probability = best_model.predict_proba(new_flower)

species_map = {0:'🌸 Setosa', 1:'🌺 Versicolor', 2:'🌻 Virginica'}
print(f"Predicted species: {species_map[prediction[0]]}")
print(f"Confidence: {max(probability[0])*100:.1f}%")

In [ ]:
import pickle

# Save model to file
with open('iris_model.pkl', 'wb') as f:
    pickle.dump(best_model, f)

print("✅ Model saved as iris_model.pkl")

# Load and test it
with open('iris_model.pkl', 'rb') as f:
    loaded_model = pickle.load(f)

test_pred = loaded_model.predict(new_flower)
print(f"Loaded model predicts: {species_map[test_pred[0]]}")